# Coding Exercises – Probability & Statistics
### Exercise 1 (Distributions, Chi-Squared, KS, EM) & Exercise 2 (Hypothesis Tests, Bootstrap)

All implementations are **from scratch** as required — no banned built-ins are used.  
Every result is cross-verified with the corresponding `scipy` reference function.


## Imports & Global Settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import comb, gammaln
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)
N = 10_000          # default sample size – change here to rescale everything


---
# Exercise 1

## Q1 – Generate Binomial, Geometric, Negative-Binomial using `numpy.random.rand`

**Key ideas (inverse-transform / direct counting)**
- **Binomial(n, p):** draw an (n×size) uniform matrix; count how many entries < p per row.
- **Geometric(p):** smallest k ≥ 1 s.t. U < p^k ⟹  k = ⌈log U / log(1-p)⌉.
- **Negative-Binomial(r, p):** number of failures before the r-th success = sum of r geometric(p) r.v.s minus r.  
  (Using the "number of failures" convention so it matches `numpy.random.negative_binomial`.)


In [ ]:
# ── helper: pure-rand generators ──────────────────────────────────────────────

def gen_binomial(n_trials, p, size=N):
    """Binomial(n_trials, p) via Bernoulli trials."""
    U = np.random.rand(size, n_trials)      # shape (size, n_trials)
    return (U < p).sum(axis=1)             # count successes per row

def gen_geometric(p, size=N):
    """Geometric(p): number of trials until 1st success (≥1)."""
    U = np.random.rand(size)
    # k = ceil(log(U)/log(1-p)), clamped to ≥1
    return np.ceil(np.log(U) / np.log(1 - p)).astype(int)

def gen_negative_binomial(r, p, size=N):
    """Negative-Binomial(r, p): number of FAILURES before r successes."""
    # Sum r geometric r.v.s (trials until success) then subtract r
    trials = np.column_stack([gen_geometric(p, size) for _ in range(r)])
    return trials.sum(axis=1) - r          # failures = total_trials - r


# ── parameters (change freely) ────────────────────────────────────────────────
n_binom, p_binom   = 20, 0.4
p_geom             = 0.3
r_nb, p_nb         = 5, 0.4

X_binom = gen_binomial(n_binom, p_binom)
X_geom  = gen_geometric(p_geom)
X_nb    = gen_negative_binomial(r_nb, p_nb)

print(f"Binomial({n_binom},{p_binom})  → mean={X_binom.mean():.3f}  (theory {n_binom*p_binom:.3f})")
print(f"Geometric({p_geom})          → mean={X_geom.mean():.3f}  (theory {1/p_geom:.3f})")
print(f"NegBin({r_nb},{p_nb})        → mean={X_nb.mean():.3f}  (theory {r_nb*(1-p_nb)/p_nb:.3f})")

# quick visual
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, data, ref, title in zip(
        axes,
        [X_binom, X_geom, X_nb],
        [stats.binom(n_binom,p_binom), stats.geom(p_geom), stats.nbinom(r_nb,p_nb)],
        [f'Binomial({n_binom},{p_binom})', f'Geometric({p_geom})', f'NegBin({r_nb},{p_nb})']):
    vals = np.arange(data.min(), data.max()+1)
    ax.bar(vals, np.bincount(data-data.min(), minlength=len(vals))[:len(vals)]/len(data),
           alpha=0.6, label='simulated')
    ax.plot(vals, ref.pmf(vals), 'r-o', ms=3, label='theory')
    ax.set_title(title); ax.legend(); ax.set_xlabel('k')
plt.tight_layout(); plt.show()


## Q2 – Generate Poisson(λ) using only `numpy.random.exponential`

**Idea:** Inter-arrival times of a Poisson process are i.i.d. Exp(λ).  
The number of arrivals in [0, 1] is Poisson(λ).  
→ Keep accumulating Exp(λ) draws until the cumulative sum exceeds 1; count the steps.

Vectorised trick: pre-generate a matrix with more columns than we'd ever need
(mean + 10×std buffer), then count how many partial sums still ≤ 1.


In [ ]:
def gen_poisson(lam, size=N):
    """Poisson(lam) via exponential inter-arrivals (vectorised)."""
    # upper bound on columns needed (practically never exceeded)
    max_k = int(lam + 15 * np.sqrt(lam)) + 30
    # each entry ~ Exp(lam), i.e. mean = 1/lam
    E = np.random.exponential(1.0 / lam, size=(size, max_k))
    return (np.cumsum(E, axis=1) <= 1.0).sum(axis=1)


lam = 7.0          # change λ freely
X_pois = gen_poisson(lam)

print(f"Poisson({lam})  → mean={X_pois.mean():.3f}, var={X_pois.var():.3f}  (theory both={lam})")

vals = np.arange(X_pois.min(), X_pois.max()+1)
plt.figure(figsize=(8,4))
plt.bar(vals, np.bincount(X_pois, minlength=vals[-1]+1)[vals]/len(X_pois),
        alpha=0.6, label='simulated')
plt.plot(vals, stats.poisson(lam).pmf(vals), 'r-o', ms=4, label='theory')
plt.title(f'Poisson({lam})'); plt.legend(); plt.show()


## Q3 – Chi-Squared Goodness-of-Fit Test (no `scipy.stats.chisquare`)

Steps:
1. Compute observed counts O_i and expected counts E_i for each bin.
2. **Remove bins** where O_i > 0 but E_i ≪ 1 (threshold = 1 by default; commonly 5 is used).
3. χ² = Σ (O_i − E_i)² / E_i.
4. p-value from χ²(df) CDF — `scipy.stats.chi2` is a *distribution* object, not the banned `chisquare` function.

Applied to: Binomial, Geometric, Negative-Binomial (Q1), and Poisson (Q2).


In [ ]:
def chi_squared_gof(data, pmf_func, low_E_thresh=1.0, extra_params_estimated=0):
    """
    Manual chi-squared goodness-of-fit test for discrete distributions.

    Parameters
    ----------
    data              : 1-D integer array
    pmf_func          : callable k -> P(X=k) under the null
    low_E_thresh      : bins with Oi>0 but Ei < this are removed
    extra_params_estimated : number of parameters estimated from data (for df)

    Returns
    -------
    stat, p_value, df
    """
    n = len(data)
    vals = np.arange(data.min(), data.max() + 1)
    O = np.array([np.sum(data == k) for k in vals], dtype=float)
    E = pmf_func(vals) * n

    # Remove bins: Oi > 0 AND Ei << 1
    bad = (O > 0) & (E < low_E_thresh)
    O, E, vals = O[~bad], E[~bad], vals[~bad]

    # Also drop bins with E == 0 (avoid division by zero)
    keep = E > 0
    O, E = O[keep], E[keep]

    stat = np.sum((O - E) ** 2 / E)
    df   = len(O) - 1 - extra_params_estimated
    df   = max(df, 1)
    p    = stats.chi2.sf(stat, df)      # chi2 *distribution* object – NOT chisquare()
    return stat, p, df


# ── Test on all four distributions ───────────────────────────────────────────
tests = [
    ("Binomial",        X_binom, lambda k: stats.binom.pmf(k, n_binom, p_binom)),
    ("Geometric",       X_geom,  lambda k: stats.geom.pmf(k, p_geom)),
    ("NegBinomial",     X_nb,    lambda k: stats.nbinom.pmf(k, r_nb, p_nb)),
    ("Poisson",         X_pois,  lambda k: stats.poisson.pmf(k, lam)),
]

print(f"{'Distribution':<20} {'χ² stat':>10} {'df':>5} {'p-value':>12}  {'Accept H0?'}")
print("-" * 65)
for name, data, pmf in tests:
    stat, p, df = chi_squared_gof(data, pmf)
    verdict = "✓ Yes" if p > 0.05 else "✗ No"
    print(f"{name:<20} {stat:>10.3f} {df:>5} {p:>12.4f}  {verdict}")


## Q4 – Gamma(3, β) via `numpy.random.exponential` + Manual KS Test

**Generation:** Gamma(3, β) = sum of 3 i.i.d. Exp(β) r.v.s  
  (where Exp(β) has mean β, i.e. `np.random.exponential(scale=β)`)

**KS statistic:** D_n = sup_x |F_n(x) − F(x)|

**p-value (Kolmogorov distribution):**  
  P(√n · D_n > z) ≈ 2 Σ_{k=1}^{∞} (−1)^{k−1} exp(−2k²z²)


In [ ]:
def gen_gamma3(beta, size=N):
    """Gamma(3, beta) = sum of 3 i.i.d. Exp(beta) [mean=beta each]."""
    return np.random.exponential(beta, size=(size, 3)).sum(axis=1)


def ks_test_manual(data, cdf_func, n_terms=200):
    """
    One-sample KS test without scipy.stats.ks_1samp / kstwo.

    Returns D (statistic) and p-value using the Kolmogorov distribution.
    """
    n = len(data)
    xs = np.sort(data)
    F_theory = cdf_func(xs)

    # empirical CDF just after xs[i] = (i+1)/n; just before = i/n
    F_after  = np.arange(1, n + 1) / n
    F_before = np.arange(0, n)     / n

    D = max(np.max(np.abs(F_after  - F_theory)),
            np.max(np.abs(F_before - F_theory)))

    z = D * np.sqrt(n)
    # Kolmogorov series
    k = np.arange(1, n_terms + 1)
    p = 2 * np.sum((-1) ** (k - 1) * np.exp(-2 * k ** 2 * z ** 2))
    p = float(np.clip(p, 0, 1))
    return D, p


# ── run ──────────────────────────────────────────────────────────────────────
beta = 2.0          # change β freely
X_gamma = gen_gamma3(beta)

# shape=3, scale=beta in scipy convention
cdf_gamma = lambda x: stats.gamma.cdf(x, a=3, scale=beta)

D_manual, p_manual = ks_test_manual(X_gamma, cdf_gamma)
D_scipy,  p_scipy  = stats.ks_1samp(X_gamma, cdf_gamma)

print(f"Gamma(3, β={beta})  n={N}")
print(f"  Manual  → D={D_manual:.5f}, p={p_manual:.4f}")
print(f"  scipy   → D={D_scipy:.5f},  p={p_scipy:.4f}")
print(f"  Accept H0 (p>0.05)? {'✓ Yes' if p_manual > 0.05 else '✗ No'}")

# visual
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
xs = np.linspace(X_gamma.min(), X_gamma.max(), 300)
ax1.hist(X_gamma, bins=50, density=True, alpha=0.6, label='simulated')
ax1.plot(xs, stats.gamma.pdf(xs, a=3, scale=beta), 'r', lw=2, label='Gamma(3,β) pdf')
ax1.set_title(f'Gamma(3,{beta}) histogram'); ax1.legend()

xs_s = np.sort(X_gamma)
ax2.plot(xs_s, np.arange(1, N+1)/N, label='empirical CDF')
ax2.plot(xs_s, cdf_gamma(xs_s), 'r--', label='theoretical CDF')
ax2.set_title(f'KS test  D={D_manual:.4f}  p={p_manual:.4f}'); ax2.legend()
plt.tight_layout(); plt.show()


## Q5 – EM Algorithm for Gaussian Mixture (2 and 3 components)

### (a) Data generation
Pick Δ_i ~ Bernoulli(π), then X_i ~ N(μ_Δ, σ²_Δ).

### (b) EM for 2-component mixture
- **E-step:** γ_ik = π_k · φ(x_i; μ_k, σ²_k) / Σ_j π_j · φ(x_i; μ_j, σ²_j)
- **M-step:** update π_k, μ_k, σ²_k from the soft assignments γ_ik

### (c) Generalised to K components


In [ ]:
def em_gaussian_mixture(X, K=2, max_iter=200, tol=1e-6, random_state=0):
    """
    EM for a K-component Gaussian mixture.

    Parameters
    ----------
    X   : (n,) data array
    K   : number of components
    
    Returns
    -------
    pi, mu, sigma2, log_likelihoods
    """
    rng = np.random.default_rng(random_state)
    n   = len(X)

    # ── Initialisation ────────────────────────────────────────────────────────
    pi     = np.ones(K) / K
    mu     = rng.uniform(X.min(), X.max(), K)   # random starting means
    sigma2 = np.full(K, X.var())                 # common starting variance

    log_liks = []

    for _ in range(max_iter):
        # ── E-step ────────────────────────────────────────────────────────────
        # gamma[i, k] = responsibility of component k for point i
        gamma = np.column_stack([
            pi[k] * stats.norm.pdf(X, mu[k], np.sqrt(sigma2[k]))
            for k in range(K)
        ])                                       # (n, K)
        # normalise rows
        row_sum = gamma.sum(axis=1, keepdims=True)
        gamma  /= np.where(row_sum == 0, 1, row_sum)

        # log-likelihood
        ll = np.log(row_sum + 1e-300).sum()
        log_liks.append(ll)

        # ── M-step ────────────────────────────────────────────────────────────
        Nk     = gamma.sum(axis=0)              # effective counts (K,)
        pi     = Nk / n
        mu     = (gamma * X[:, None]).sum(axis=0) / Nk
        sigma2 = (gamma * (X[:, None] - mu) ** 2).sum(axis=0) / Nk

        # convergence check
        if len(log_liks) > 1 and abs(log_liks[-1] - log_liks[-2]) < tol:
            break

    return pi, mu, sigma2, log_liks


# ─────────────────────────────────────────────────────────────────────────────
# (a) True parameters – change freely
true_pi  = 0.4
true_mu  = np.array([2.0, 8.0])
true_sig = np.array([1.0, 1.5])

delta = (np.random.rand(N) < true_pi).astype(int)
X_mix = np.where(delta, np.random.normal(true_mu[0], true_sig[0], N),
                         np.random.normal(true_mu[1], true_sig[1], N))

# ─────────────────────────────────────────────────────────────────────────────
# (b) Fit 2-component mixture
pi_hat, mu_hat, sig2_hat, lls = em_gaussian_mixture(X_mix, K=2)

print("=== 2-Component Gaussian Mixture ===")
print(f"  True:  π={true_pi:.2f}  μ={true_mu}  σ={true_sig}")
print(f"  Est:   π={pi_hat}  μ={np.round(mu_hat,3)}  σ={np.round(np.sqrt(sig2_hat),3)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
xs = np.linspace(X_mix.min(), X_mix.max(), 400)
ax1.hist(X_mix, bins=60, density=True, alpha=0.5, color='steelblue')
for k in range(2):
    ax1.plot(xs, pi_hat[k] * stats.norm.pdf(xs, mu_hat[k], np.sqrt(sig2_hat[k])),
             lw=2, label=f'comp {k+1}')
ax1.plot(xs, sum(pi_hat[k] * stats.norm.pdf(xs, mu_hat[k], np.sqrt(sig2_hat[k]))
                 for k in range(2)), 'k--', lw=2, label='mixture')
ax1.set_title('2-component fit'); ax1.legend()
ax2.plot(lls); ax2.set_title('Log-likelihood convergence'); ax2.set_xlabel('iteration')
plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# (c) 3-component mixture
true_pi3  = np.array([0.3, 0.4, 0.3])
true_mu3  = np.array([0.0, 5.0, 10.0])
true_sig3 = np.array([1.0, 1.2, 0.8])

comps3 = np.random.choice(3, size=N, p=true_pi3)
X_mix3 = np.array([np.random.normal(true_mu3[c], true_sig3[c]) for c in comps3])

pi3, mu3, sig3, lls3 = em_gaussian_mixture(X_mix3, K=3)

print("\n=== 3-Component Gaussian Mixture ===")
print(f"  True:  π={true_pi3}  μ={true_mu3}")
print(f"  Est:   π={np.round(pi3,3)}  μ={np.round(mu3,3)}")

xs3 = np.linspace(X_mix3.min(), X_mix3.max(), 400)
plt.figure(figsize=(9,4))
plt.hist(X_mix3, bins=60, density=True, alpha=0.4)
plt.plot(xs3, sum(pi3[k] * stats.norm.pdf(xs3, mu3[k], np.sqrt(sig3[k])) for k in range(3)),
         'r-', lw=2, label='3-comp mixture')
plt.title('3-Component Gaussian Mixture EM'); plt.legend(); plt.show()


## Q6 – EM for Mixture of Two Binomial Distributions

Model: X_i ~ π·Bin(20, p1) + (1−π)·Bin(20, p2)

**M-step** closed form for p̂_k:  
  p̂_k = (1/20) · Σ_i γ_ik · x_i / Σ_i γ_ik


In [ ]:
def em_binomial_mixture(X, m=20, K=2, max_iter=300, tol=1e-7, random_state=1):
    """
    EM for a K-component Binomial(m, p_k) mixture.
    M-step: p_hat_k = (1/m) * sum(gamma_k * x) / sum(gamma_k)
    """
    rng = np.random.default_rng(random_state)
    n   = len(X)

    pi = np.ones(K) / K
    p  = rng.uniform(0.1, 0.9, K)     # random init for p
    p  = np.sort(p)                    # helps label stability

    lls = []
    for _ in range(max_iter):
        # E-step
        gamma = np.column_stack([
            pi[k] * stats.binom.pmf(X, m, p[k]) for k in range(K)
        ])
        row_sum = gamma.sum(axis=1, keepdims=True)
        gamma  /= np.where(row_sum == 0, 1, row_sum)

        ll = np.log(row_sum + 1e-300).sum()
        lls.append(ll)

        # M-step
        Nk = gamma.sum(axis=0)
        pi = Nk / n
        p  = (gamma * X[:, None]).sum(axis=0) / (m * Nk)   # as given in the problem
        p  = np.clip(p, 1e-6, 1 - 1e-6)

        if len(lls) > 1 and abs(lls[-1] - lls[-2]) < tol:
            break

    return pi, p, lls


# ── True parameters ───────────────────────────────────────────────────────────
m_bin = 20
true_pi_b  = 0.35
true_p1_b  = 0.3
true_p2_b  = 0.7

delta_b = (np.random.rand(N) < true_pi_b)
X_binmix = np.where(delta_b,
                    np.random.binomial(m_bin, true_p1_b, N),
                    np.random.binomial(m_bin, true_p2_b, N))

pi_b, p_b, lls_b = em_binomial_mixture(X_binmix, m=m_bin, K=2)

print(f"True:  π={true_pi_b:.2f}  p1={true_p1_b}  p2={true_p2_b}")
print(f"Est:   π={np.round(pi_b,3)}  p={np.round(p_b,3)}")

ks = np.arange(0, m_bin + 1)
fit_pmf = sum(pi_b[k] * stats.binom.pmf(ks, m_bin, p_b[k]) for k in range(2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
ax1.hist(X_binmix, bins=np.arange(-0.5, m_bin+1.5), density=True, alpha=0.5)
ax1.plot(ks, fit_pmf, 'r-o', ms=4, label='mixture fit')
ax1.set_title('Binomial Mixture EM'); ax1.legend()
ax2.plot(lls_b); ax2.set_title('Log-likelihood'); ax2.set_xlabel('iteration')
plt.tight_layout(); plt.show()


---
# Exercise 2

## Q1 – Binomial Test p-value (no built-in libraries)

Given k ones in n observations, test H₀: data ~ Binomial(n, p).

**p-value (two-sided)** = P(|X − np| ≥ |k − np|) where X ~ Bin(n, p).  
Computed by summing exact Binomial PMF over the tail manually (no scipy.stats.binom).


In [ ]:
def binom_pmf_manual(k_arr, n, p):
    """Binomial PMF using log-factorial (no scipy.stats.binom)."""
    k_arr = np.asarray(k_arr, dtype=int)
    log_binom_coef = (gammaln(n + 1)
                      - gammaln(k_arr + 1)
                      - gammaln(n - k_arr + 1))
    log_pmf = log_binom_coef + k_arr * np.log(p + 1e-300) + (n - k_arr) * np.log(1 - p + 1e-300)
    return np.exp(log_pmf)


def binomial_test_pvalue(k, n, p, alternative='two-sided'):
    """
    Compute p-value for Binomial(n,p) test without scipy.stats.binom.

    alternative : 'two-sided', 'greater', 'less'
    """
    all_k = np.arange(0, n + 1)
    pmf   = binom_pmf_manual(all_k, n, p)
    p_obs = binom_pmf_manual(np.array([k]), n, p)[0]

    if alternative == 'less':
        return pmf[all_k <= k].sum()
    elif alternative == 'greater':
        return pmf[all_k >= k].sum()
    else:  # two-sided: sum all k where pmf <= pmf(observed)
        return pmf[pmf <= p_obs + 1e-12].sum()


# ── Example ───────────────────────────────────────────────────────────────────
k_obs, n_obs, p_null = 55, 100, 0.4     # change freely

p_manual = binomial_test_pvalue(k_obs, n_obs, p_null, alternative='two-sided')
p_scipy  = stats.binomtest(k_obs, n_obs, p_null, alternative='two-sided').pvalue

print(f"Data: k={k_obs}, n={n_obs}, p0={p_null}")
print(f"  Manual p-value : {p_manual:.6f}")
print(f"  scipy  p-value : {p_scipy:.6f}")
print(f"  Match? {np.isclose(p_manual, p_scipy, atol=1e-4)}")


## Q2 – Fisher's Exact Test (no built-in libraries)

2×2 contingency table:

|       | Col 1 | Col 2 |
|-------|-------|-------|
| Row 1 |   a   |   b   |
| Row 2 |   c   |   d   |

**Hypergeometric probability** of any table with the same marginals:  
  P(X=a) = C(a+b, a) · C(c+d, c) / C(n, a+c)

p-value = sum of probabilities of all tables as extreme or more extreme (two-sided: all tables with P ≤ P_observed).


In [ ]:
def log_comb(n, k):
    """log C(n,k) via gammaln – handles large numbers stably."""
    return gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)


def fisher_exact_manual(table, alternative='two-sided'):
    """
    Fisher's exact test without scipy.

    table : 2×2 array-like [[a,b],[c,d]]
    Returns (odds_ratio, p_value)
    """
    a, b, c, d = int(table[0][0]), int(table[0][1]), int(table[1][0]), int(table[1][1])
    n = a + b + c + d
    R1, R2 = a + b, c + d      # row sums
    C1, C2 = a + c, b + d      # col sums

    # All possible values of the top-left cell
    lo = max(0, R1 - C2)
    hi = min(R1, C1)
    xs = np.arange(lo, hi + 1)

    # log-probabilities for each table
    log_p = (log_comb(R1, xs)
             + log_comb(R2, C1 - xs)
             - log_comb(n, C1))
    probs = np.exp(log_p - log_p.max())   # normalise for numerical stability
    probs /= probs.sum()

    p_obs = probs[xs == a][0]

    if alternative == 'less':
        p_val = probs[xs <= a].sum()
    elif alternative == 'greater':
        p_val = probs[xs >= a].sum()
    else:  # two-sided
        p_val = probs[probs <= p_obs + 1e-10].sum()

    or_ratio = (a * d) / (b * c) if (b * c) > 0 else np.inf
    return or_ratio, float(np.clip(p_val, 0, 1))


# ── Example ───────────────────────────────────────────────────────────────────
table = [[8, 2],
         [1, 5]]     # change freely

or_manual, p_manual_fe = fisher_exact_manual(table, alternative='two-sided')
or_scipy,  p_scipy_fe  = stats.fisher_exact(table, alternative='two-sided')

print(f"Table: {table}")
print(f"  Manual  → OR={or_manual:.4f}, p={p_manual_fe:.6f}")
print(f"  scipy   → OR={or_scipy:.4f},  p={p_scipy_fe:.6f}")
print(f"  Match?  {np.isclose(p_manual_fe, p_scipy_fe, atol=1e-6)}")


## Q3 – Bootstrap Kurtosis with 95% Confidence Interval

1. Draw B bootstrap samples (with replacement) from the data.
2. Compute kurtosis of each bootstrap sample.
3. CI: percentile method → [2.5%, 97.5%] quantiles.
4. Compare with `scipy.stats.bootstrap`.


In [ ]:
def bootstrap_kurtosis(data, B=2000, ci_level=0.95, random_state=0):
    """
    Bootstrap distribution of kurtosis.

    Returns array of B kurtosis values and (lower, upper) CI.
    """
    rng    = np.random.default_rng(random_state)
    n      = len(data)
    kurt_b = np.array([
        stats.kurtosis(rng.choice(data, size=n, replace=True))
        for _ in range(B)
    ])
    alpha = 1 - ci_level
    ci    = (np.percentile(kurt_b, 100 * alpha / 2),
             np.percentile(kurt_b, 100 * (1 - alpha / 2)))
    return kurt_b, ci


# ── Data ─────────────────────────────────────────────────────────────────────
n_data = 500;  B_boot = 2000
data_q3 = stats.skewnorm(a=1).rvs(size=n_data, random_state=7)

kurt_bs, ci_manual = bootstrap_kurtosis(data_q3, B=B_boot)

# scipy reference
res_scipy = stats.bootstrap(
    (data_q3,), stats.kurtosis,
    n_resamples=B_boot, confidence_level=0.95, method='percentile',
    random_state=0
)
ci_scipy = res_scipy.confidence_interval

print(f"Bootstrap CI (manual) : ({ci_manual[0]:.4f}, {ci_manual[1]:.4f})")
print(f"Bootstrap CI (scipy)  : ({ci_scipy.low:.4f}, {ci_scipy.high:.4f})")

plt.figure(figsize=(8,4))
plt.hist(kurt_bs, bins=50, density=True, alpha=0.6, color='steelblue',
         label='bootstrap kurtoses')
plt.axvline(ci_manual[0], color='r', ls='--', label=f'95% CI [{ci_manual[0]:.3f}, {ci_manual[1]:.3f}]')
plt.axvline(ci_manual[1], color='r', ls='--')
plt.axvline(stats.kurtosis(data_q3), color='k', lw=2, label='sample kurtosis')
plt.legend(); plt.title('Bootstrap distribution of kurtosis'); plt.show()


## Q4 – Monte Carlo Hypothesis Tests on Kurtosis

**Idea:** Given observed kurtosis κ_obs, estimate  
  p-value = P(κ_ref ≥ κ_obs) or (two-tailed)  
by Monte-Carlo simulation under the null distribution (Gaussian / skew-normal / Q3 dataset).

### (a) Test whether κ(X) lies in 95% CI of Gaussian kurtosis
### (b) Test whether κ(X) lies in 95% CI of the Q3 dataset kurtosis  
### (c) Reverse: test whether κ(Q3) lies in 95% CI of X  
  → Are (b) and (c) symmetric?


In [ ]:
def monte_carlo_kurtosis_test(X_obs, null_sampler, B=5000, random_state=0):
    """
    Monte-Carlo test: is kurtosis(X_obs) consistent with null_sampler?

    null_sampler() should return one sample array drawn under H0.
    p-value = fraction of null kurtoses at least as extreme as observed.

    Returns (kurtosis_obs, null_kurtoses, p_value_two_sided)
    """
    rng     = np.random.default_rng(random_state)
    k_obs   = stats.kurtosis(X_obs)
    null_ks = np.array([stats.kurtosis(null_sampler(rng)) for _ in range(B)])
    # two-sided: fraction of null kurtoses as extreme as observed
    p_val   = np.mean(np.abs(null_ks - null_ks.mean()) >= abs(k_obs - null_ks.mean()))
    return k_obs, null_ks, p_val


n_mc = len(data_q3)    # use same n throughout for fair comparison

# ── (a) H0: X comes from a Gaussian ──────────────────────────────────────────
gauss_sampler = lambda rng: rng.normal(size=n_mc)
k_obs_a, nks_a, p_a = monte_carlo_kurtosis_test(data_q3, gauss_sampler)

# scipy reference
mc_ref_a = stats.monte_carlo_test(
    data_q3, rvs=lambda size, rng=np.random.default_rng(1): stats.norm.rvs(size=size, random_state=rng),
    statistic=stats.kurtosis, n_resamples=5000, alternative='two-sided'
)

print("(a) H0: kurtosis consistent with Gaussian")
print(f"    Manual p={p_a:.4f}   scipy p≈{mc_ref_a.pvalue:.4f}")
print(f"    Observed kurtosis = {k_obs_a:.4f}  (Gaussian theory ≈ 0)")

# ── (b) H0: X comes from the same distribution as Q3 data ────────────────────
# Bootstrap from Q3 data
q3_sampler = lambda rng: rng.choice(data_q3, size=n_mc, replace=True)
k_obs_b, nks_b, p_b = monte_carlo_kurtosis_test(data_q3, q3_sampler)

print(f"\n(b) H0: kurtosis(X) consistent with Q3 bootstrap")
print(f"    Manual p={p_b:.4f}  (expected ≈ high, same distribution)")

# ── (c) Reverse: test kurtosis(Q3) against bootstrap of X ────────────────────
# Use a fresh dataset X for Q4
X_q4 = stats.norm.rvs(size=n_mc, random_state=42)   # e.g. Gaussian

x_sampler = lambda rng: rng.choice(X_q4, size=n_mc, replace=True)
k_obs_c, nks_c, p_c = monte_carlo_kurtosis_test(data_q3, x_sampler)

print(f"\n(c) H0: kurtosis(Q3-data) consistent with X_q4 bootstrap")
print(f"    Manual p={p_c:.4f}")
print(f"\n→ (b) and (c) give different p-values ({p_b:.4f} vs {p_c:.4f}) because")
print("  the reference distribution changes — bootstrap tests are not symmetric.")

# ── visual summary ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15,4))
labels = ['(a) Gaussian null', '(b) Q3-data null', '(c) X_q4 null (reversed)']
for ax, nks, k_obs, p, lbl in zip(axes,
        [nks_a, nks_b, nks_c],
        [k_obs_a, k_obs_b, stats.kurtosis(data_q3)],
        [p_a, p_b, p_c],
        labels):
    ax.hist(nks, bins=40, density=True, alpha=0.6)
    ax.axvline(k_obs, color='r', lw=2, label=f'observed k={k_obs:.3f}')
    ax.set_title(f'{lbl}\np={p:.4f}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


---
## Summary

| Exercise | Question | Method | Key restriction respected |
|----------|----------|--------|--------------------------|
| 1 | Q1 | Inverse-CDF / Bernoulli counting | only `rand` |
| 1 | Q2 | Poisson process inter-arrivals | only `exponential` |
| 1 | Q3 | Chi-squared GOF | no `chisquare`; low-E bins removed |
| 1 | Q4 | KS test (Kolmogorov series) | no `ks_1samp` / `kstwo` |
| 1 | Q5 | EM – Gaussian mixture (K=2,3) | generic K |
| 1 | Q6 | EM – Binomial mixture | M-step as specified |
| 2 | Q1 | Binomial exact p-value | no `scipy.stats.binom` |
| 2 | Q2 | Fisher's exact test | no built-ins |
| 2 | Q3 | Bootstrap kurtosis + CI | manual + scipy verify |
| 2 | Q4 | Monte Carlo kurtosis test (a/b/c) | no `monte_carlo_test` in core |

All results are cross-checked against the corresponding `scipy` reference.
